<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/slap2_fourparadigm_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Paradigm-matched SLAP2 — first characterization across four error types (PRELIMINARY)

The SLAP2 dendritic-glutamate dataset (DANDI **001424**, iGluSnFR) originally stored stimuli
as a single monolithic gratings stream. **8 newer sessions** (subjects 828408 / 828409 / 829704)
now use the **same named-block design** as the Neuropixels and mesoscope datasets — 2 sessions
each of standard-oddball, sequence, duration, and sensorimotor. This notebook computes, for the
first time, the prediction-error index in each paradigm at dendritic-glutamate resolution and
compares it to the Neuropixels (spiking) reference.

> **This is preliminary: n = 2 sessions per paradigm.** A null with 2 sessions could be a real
> modality difference (imaging shows weak/absent oddball effects — see Result 2) or simply
> underpowered. We report it as a first look, not a confirmatory result. The sensorimotor
> contrast is the weakest: only ~5 open-loop running events per session (same locomotion limit
> as Result 3).

In [ ]:
import sys, subprocess
try:
    import pynwb, remfile, h5py  # noqa
except ImportError:
    subprocess.run([sys.executable,"-m","pip","install","-q","pynwb","remfile","h5py","requests","pandas","numpy","matplotlib","scipy"],check=True)
import numpy as np, pandas as pd, h5py, remfile, requests
from scipy import stats as ss

def resolve_asset(subject, date_or_ses, dandiset="001424"):
    """SLAP2 sessions: resolve by (subject, session-id substring) via path. IDs are not stable."""
    u=f"https://api.dandiarchive.org/api/dandisets/{dandiset}/versions/draft/assets/"
    r=requests.get(u,params={"path":f"sub-{subject}/"},timeout=30).json()
    hits=[a for a in r["results"] if date_or_ses in a["path"]]
    if len(hits)!=1: raise LookupError(f"{subject}/{date_or_ses}: {len(hits)} matches")
    return hits[0]["asset_id"]
def s3(aid,ds="001424"):
    return requests.get(f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/draft/assets/{aid}/download/",allow_redirects=False,timeout=60).headers["Location"]
def dec(a): return np.array([x.decode() if isinstance(x,bytes) else x for x in a])

In [ ]:
# The 8 paradigm-matched SLAP2 sessions (subject, session-id substring, block name)
SESSIONS = {
  "standard_oddball": [("828408","2025-11-18","standard_oddball"), ("829704","2025-12-11","standard_oddball")],
  "sequence":         [("829704","2025-12-16","sequential_oddball"), ("828409","2025-11-20","sequential_oddball")],
  "duration":         [("829704","2025-12-18","jitter_oddball"), ("828409","2025-11-21","jitter_oddball")],
  "sensorimotor":     [("829704","2025-12-10","motor_oddball"), ("828408","2025-11-13","motor_oddball")],
}
def load_dff(fh,dmd):
    fl=fh["processing"]["ophys"][f"Fluorescence_{dmd}"][f"{dmd}_dFF_green"]
    return fl["data"], fl["timestamps"][:]
def win_resp(dff,tt,onsets,rw=(0.0,0.5),bw=(-0.3,-0.05)):
    if len(onsets)==0: return np.full(dff.shape[1],np.nan)
    R=np.full((len(onsets),dff.shape[1]),np.nan)
    for i,o in enumerate(onsets):
        l0=np.searchsorted(tt,o+rw[0]);h0=np.searchsorted(tt,o+rw[1])
        lb=np.searchsorted(tt,o+bw[0]);hb=np.searchsorted(tt,o+bw[1])
        if h0>l0 and hb>lb: R[i]=np.nanmean(dff[l0:h0],0)-np.nanmean(dff[lb:hb],0)
    return np.nanmean(R,axis=0)
def dvi_pair(a,b):
    m=~(np.isnan(a)|np.isnan(b)); return (a[m]-b[m])/(np.abs(a[m])+np.abs(b[m])+1e-9)

## Per-paradigm prediction-error index (each vs. Neuropixels reference)

In [ ]:
def extract(paradigm, subj, ses, blk):
    fh=h5py.File(remfile.File(s3(resolve_asset(subj,ses))),"r")
    g=fh["intervals"][blk]; TT=dec(g["TrialType"][:]); ts=g["start_time"][:]; iv=fh["intervals"]
    rows=[]
    if paradigm in ("standard_oddball","sequence"):
        cblk="standard_control" if paradigm=="standard_oddball" else "sequential_control_block"
        gc=iv[cblk]; cori=dec(gc["Orientation"][:]).astype(float); cts=gc["start_time"][:]
        if paradigm=="sequence":
            tis=dec(g["TrialInSequence"][:]).astype(float); dev=ts[(TT=="orientation_90")&(tis==3)]
        else:
            dev=ts[TT=="orientation_90"]
        ctrl=cts[np.abs(np.degrees(cori)-90)<5]
        for dmd in ["DMD1","DMD2"]:
            try: dff,tt=load_dff(fh,dmd)
            except: continue
            for v in dvi_pair(win_resp(dff,tt,dev),win_resp(dff,tt,ctrl)): rows.append(v)
    elif paradigm=="duration":
        omis=ts[TT=="omission"]; std=ts[TT=="standard"][:400]
        for dmd in ["DMD1","DMD2"]:
            try: dff,tt=load_dff(fh,dmd)
            except: continue
            om=win_resp(dff,tt,omis,rw=(0.0,0.15),bw=(-0.25,-0.10))
            sr=win_resp(dff,tt,std,rw=(0.0,0.15),bw=(-0.25,-0.10))
            m=~(np.isnan(om)|np.isnan(sr))
            for v in om[m]/(np.abs(om[m])+np.abs(sr[m])+1e-9): rows.append(v)
    else:  # sensorimotor: closed-run - open-run, orientation_90
        ol=iv["open_loop_prerecorded"]; oltt=dec(ol["TrialType"][:]); olst=ol["start_time"][:]
        rs=fh["processing"]["running"]; rk=[k for k in rs.keys() if "speed" in k.lower()]
        spd=rs[rk[0]]["data"][:].astype(float); rtt=rs[rk[0]]["timestamps"][:].astype(float)
        def gate(times):
            keep=[]
            for t in times:
                lo=np.searchsorted(rtt,t-1.0);hi=np.searchsorted(rtt,t)
                if hi>lo and np.abs(spd[lo:hi]).mean()>1.0: keep.append(t)
            return np.array(keep)
        cl=gate(ts[TT=="motor_orientation_90"]); op=gate(olst[oltt=="motor_orientation_90"])
        for dmd in ["DMD1","DMD2"]:
            try: dff,tt=load_dff(fh,dmd)
            except: continue
            for v in dvi_pair(win_resp(dff,tt,cl),win_resp(dff,tt,op)): rows.append(v)
    fh.close(); return rows

RES={}
for para,sess in SESSIONS.items():
    vals=[]
    for subj,ses,blk in sess:
        vals+= [(subj,v) for v in extract(para,subj,ses,blk)]
        print(f"  {para}/{subj}: n_roi_so_far={len(vals)}")
    RES[para]=vals

In [ ]:
# Summary vs Neuropixels reference.
# n=2 sessions/paradigm: a bootstrap that resamples ROIs within a fixed subject set omits
# between-animal variance (the dominant term at n=2) and gives CIs that are far too narrow.
# The honest report at this n is the TWO PER-SESSION MEDIANS, not a CI — so we show those and
# a session-resampled CI only for context (it is deliberately wide/uninformative at n=2).
ECE={"standard_oddball":0.454,"sequence":0.205,"duration":0.321,"sensorimotor":0.096}
def session_ci(D):
    # resample SESSIONS (subjects) with replacement, then ROIs within each drawn session
    rng=np.random.default_rng(42); subs=D.subject.unique()
    by={s:D[D.subject==s].v.values for s in subs}
    meds=[]
    for _ in range(5000):
        drawn=rng.choice(subs,len(subs),replace=True)
        pool=np.concatenate([rng.choice(by[s],len(by[s]),replace=True) for s in drawn if len(by[s])])
        meds.append(np.median(pool))
    return np.percentile(meds,[2.5,97.5])
summary=[]
for para,vals in RES.items():
    D=pd.DataFrame(vals,columns=["subject","v"]); med=D.v.median()
    lo,hi=session_ci(D)
    persess=D.groupby("subject").v.median().round(3).to_dict()
    summary.append(dict(paradigm=para,slap2_median=med,ci_lo_session=lo,ci_hi_session=hi,
                        n_roi=len(D),n_sess=D.subject.nunique(),
                        session_medians=str(persess),ece_ref=ECE[para]))
    print(f"{para:18}: SLAP2 median {med:+.3f}  per-session {persess}  "
          f"(session-resampled CI[{lo:+.3f},{hi:+.3f}], wide at n=2)  vs NP {ECE[para]:+.2f}")
pd.DataFrame(summary).to_csv("slap2_fourparadigm_summary.csv",index=False)
pd.DataFrame(summary)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import ast
order=["standard_oddball","sequence","duration","sensorimotor"]
lab={"standard_oddball":"Feature-oddball","sequence":"Sequence","duration":"Duration/timing","sensorimotor":"Sensorimotor"}
fig,ax=plt.subplots(figsize=(9.5,4.9)); fig.subplots_adjust(left=0.19,right=0.975,top=0.86,bottom=0.22)
yy=np.arange(len(order))[::-1]
S={d["paradigm"]:d for d in summary}
for y,p in zip(yy,order):
    r=S[p]
    # Neuropixels reference (open grey)
    ax.plot(r["ece_ref"],y+0.20,"o",color="0.5",ms=8,mfc="white",mew=1.5)
    # SLAP2: plot the TWO PER-SESSION medians as individual points (honest at n=2),
    # plus the wide session-resampled CI as a faint line for context.
    sm=ast.literal_eval(r["session_medians"])
    ax.plot([r["ci_lo_session"],r["ci_hi_session"]],[y-0.18,y-0.18],color="#7d3c98",lw=1.5,alpha=0.35,solid_capstyle="round")
    for v in sm.values():
        ax.plot(v,y-0.18,"o",color="#7d3c98",ms=8,mfc="#7d3c98",alpha=0.85)
    ax.text(-0.62,y,lab[p],va="center",fontsize=9.5,fontweight="bold")
ax.axvline(0,color="k",lw=0.8,ls=":"); ax.set_yticks([]); ax.set_xlim(-0.62,0.62); ax.set_ylim(-0.6,3.4)
ax.set_xlabel("prediction-error index  (-1...+1)")
for sp in ["left","top","right"]: ax.spines[sp].set_visible(False)
ax.legend(handles=[Line2D([0],[0],marker="o",color="0.5",mfc="white",mew=1.5,ls="none",ms=8,label="Neuropixels (reference, spiking)"),
                   Line2D([0],[0],marker="o",color="#7d3c98",ls="none",ms=8,label="SLAP2 per-session median (glutamate, 2 sessions)")],
          frameon=False,fontsize=7.5,loc="upper center",bbox_to_anchor=(0.5,-0.16),ncol=2)
ax.set_title("Paradigm-matched SLAP2 (n=2/paradigm): per-session medians vs the Neuropixels reference\n"
             "(2 sessions is too few for a meaningful CI; the faint bar is a session-resampled interval for context only)",
             fontsize=8,loc="left")
fig.savefig("slap2_fourparadigm.png",dpi=185,bbox_inches="tight"); plt.show()

### Takeaway

At n = 2 sessions per paradigm, **none of the four SLAP2 paradigms reproduces the positive
prediction-error index seen in Neuropixels spiking** — all sit at or below zero. This extends
the Result 2 cross-scale finding (imaging modalities show weak/absent oddball effects) to the
dendritic-glutamate scale and to the non-oddball paradigms.

**Interpretation is deliberately limited.** Two candidate explanations — (i) a genuine
dissociation between input glutamate (iGluSnFR, dendritic) and somatic spiking output, or
(ii) simple underpowering at n = 2 — cannot be separated with this sample. The dataset is
actively growing (more SLAP2 sessions arrived mid-2026); this analysis should be re-run as the
paradigm-matched SLAP2 set expands.